# 06 · Graph Attention Networks (GAT)

## What this notebook covers
GAT (Veličković et al. 2018) introduces attention into the message passing framework. Instead of aggregating neighbours with uniform weights, GAT learns *how much attention to pay to each neighbour*.

## The attention mechanism
For each pair of connected nodes (i, j), GAT computes an attention coefficient:

    e_ij = LeakyReLU(a^T [W h_i || W h_j])

Then normalises via softmax over the neighbourhood:

    α_ij = softmax_j(e_ij) = exp(e_ij) / Σ_{k ∈ N(i)} exp(e_ik)

The aggregated message is:

    h_i' = σ(Σ_{j ∈ N(i)} α_ij W h_j)

**Multi-head attention**: GAT runs K independent attention heads and concatenates (or averages) their outputs — same as transformer multi-head attention but over graph neighbourhoods.

## When GAT outperforms GCN/GraphSAGE
GAT is particularly strong when:
- Neighbour importance is heterogeneous (some neighbours are much more informative than others)
- The graph has noisy edges
- Interpretability matters — attention scores can visualise which connections are important

For homogeneous graphs (all neighbours roughly equally informative), GCN/GraphSAGE often match GAT at lower computational cost.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
torch.manual_seed(42); np.random.seed(42)

try:
    from torch_geometric.datasets import Planetoid
    from torch_geometric.nn import GATConv
    HAS_PYG = True
except ImportError:
    HAS_PYG = False

In [ ]:
if HAS_PYG:
    dataset = Planetoid(root="/tmp/Cora", name="Cora")
    data = dataset[0]

    class GAT(torch.nn.Module):
        def __init__(self, in_channels, hidden_channels, out_channels, heads=8):
            super().__init__()
            self.conv1 = GATConv(in_channels, hidden_channels, heads=heads, dropout=0.6)
            self.conv2 = GATConv(hidden_channels * heads, out_channels, heads=1, concat=False, dropout=0.6)

        def forward(self, x, edge_index):
            x = F.dropout(x, p=0.6, training=self.training)
            x = F.elu(self.conv1(x, edge_index))
            x = F.dropout(x, p=0.6, training=self.training)
            return self.conv2(x, edge_index)

    gat = GAT(dataset.num_features, 8, dataset.num_classes, heads=8)
    opt = torch.optim.Adam(gat.parameters(), lr=0.005, weight_decay=5e-4)

    best_val, best_test = 0, 0
    for epoch in range(200):
        gat.train()
        opt.zero_grad()
        out = gat(data.x, data.edge_index)
        F.cross_entropy(out[data.train_mask], data.y[data.train_mask]).backward()
        opt.step()
        gat.eval()
        with torch.no_grad():
            pred = out.argmax(dim=1)
            val  = (pred[data.val_mask]  == data.y[data.val_mask]).float().mean().item()
            test = (pred[data.test_mask] == data.y[data.test_mask]).float().mean().item()
        if val > best_val: best_val, best_test = val, test

    print(f"GAT on Cora: Test accuracy = {best_test:.4f}")
    print("(Typical: ~83% vs GCN ~81% — attention helps on citation networks)")
else:
    print("GAT on Cora typically achieves ~83% test accuracy.")

In [ ]:
# ── GAT from scratch: single-head attention ────────────────────────────────────
G = nx.karate_club_graph()
n = G.number_of_nodes()
edge_list = list(G.edges()) + [(v,u) for u,v in G.edges()] + [(i,i) for i in range(n)]
edges = torch.LongTensor(edge_list).T
X_kc = torch.eye(n)
y_kc = torch.LongTensor([0 if G.nodes[i]["club"]=="Mr. Hi" else 1 for i in range(n)])

class GATLayer(nn.Module):
    def __init__(self, in_dim, out_dim, n_heads=4):
        super().__init__()
        self.n_heads = n_heads
        self.out_per_head = out_dim // n_heads
        self.W = nn.Linear(in_dim, out_dim, bias=False)
        self.a = nn.Parameter(torch.randn(n_heads, 2 * self.out_per_head))
        nn.init.xavier_uniform_(self.a.data)

    def forward(self, X, edge_index):
        H = self.W(X).view(X.size(0), self.n_heads, self.out_per_head)  # (N, K, d)
        src, tgt = edge_index[0], edge_index[1]
        h_cat = torch.cat([H[src], H[tgt]], dim=-1)  # (E, K, 2d)
        e = F.leaky_relu((h_cat * self.a).sum(dim=-1), negative_slope=0.2)  # (E, K)
        # Softmax per target node per head
        alpha = torch.zeros(X.size(0), X.size(0), self.n_heads)
        for k in range(self.n_heads):
            alpha_dense = torch.full((n, n), float("-inf"))
            for idx in range(edge_index.size(1)):
                alpha_dense[tgt[idx], src[idx]] = e[idx, k]
            alpha[:, :, k] = torch.softmax(alpha_dense, dim=1)
        # Aggregate
        out = torch.zeros(X.size(0), self.n_heads, self.out_per_head)
        for k in range(self.n_heads):
            out[:, k, :] = alpha[:, :, k] @ H[:, k, :]
        return F.elu(out.view(X.size(0), -1)), alpha

gat_layer = GATLayer(n, 16, n_heads=4)
out_emb, attn = gat_layer(X_kc, edges)
print(f"GAT layer output: {out_emb.shape}  |  Attention tensor: {attn.shape}")

In [ ]:
# Visualise attention on karate club graph
pos = nx.spring_layout(G, seed=42)
attn_np = attn[:, :, 0].detach().numpy()  # head 0

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Attention heatmap
im = axes[0].imshow(attn_np[:15, :15], cmap="Blues", aspect="auto")
plt.colorbar(im, ax=axes[0])
axes[0].set_title("Attention weights (head 0, first 15 nodes)")
axes[0].set_xlabel("Source node"); axes[0].set_ylabel("Target node")

# Graph with edge weights from attention
edge_weights = [attn_np[v, u] * 10 for u, v in G.edges()]
nx.draw(G, pos=pos, ax=axes[1],
        node_color=["steelblue" if G.nodes[i]["club"]=="Mr. Hi" else "tomato" for i in G.nodes()],
        with_labels=True, font_size=7, node_size=200,
        width=edge_weights, edge_color="darkgrey", alpha=0.8)
axes[1].set_title("Graph with attention-weighted edges (head 0)")

plt.tight_layout()
plt.savefig(f"{base}/06_gat.png", dpi=120)
plt.show()